### Cricket clips on Colab

Set the runtime to a T4 GPU and run the cells. The zip downloads by itself.

This loads `Qwen2.5-Omni-3B` in 4-bit. Each window is 8 frames plus the commentary.

A window starts on a shot cut or on the end of a commentary pause, and it stops on a later cut or pause. One pass stops at 30 seconds. Another stops at 45 seconds. Anything shorter than 12 seconds is dropped.

The model only scores a window built around a soundtrack surge that the commentary also names. A surge is a 2-second stretch at least 1.25 times as loud as the surrounding 20 seconds. Parakeet transcribes the soundtrack once. A surge is kept when a word such as four, six, boundary, wicket, catch, or century falls between 8 seconds before it and 12 seconds after it. Surges within 20 seconds keep the louder one. Each kept surge produces one clip up to 30 seconds and one up to 45 seconds. The saved JSON keeps that set, and a second set with overlaps removed.


In [1]:
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

url = "https://sheet-importantly-parents-saturn.trycloudflare.com/videostories_colab.zip"
dest = Path("/content") if Path("/content").is_dir() else Path.cwd()
z = dest / "videostories_colab.zip"
cricket = dest / "data/india_cricket"
ready = (dest / "requirements.txt").exists() and any(cricket.glob("3.*.mp4"))

def progress(block, block_size, total):
    got = block * block_size
    mb = int(got / 5e6)
    if mb == progress.seen:
        return
    progress.seen = mb
    if total:
        print(f"download {min(got, total)/1e6:.0f} / {total/1e6:.0f} MB", flush=True)
    else:
        print(f"download {got/1e6:.0f} MB", flush=True)

progress.seen = -1
if ready:
    print("files already present", dest)
else:
    print("downloading zip", flush=True)
    partial = dest / "videostories_colab.partial"
    urlretrieve(url, partial, progress)
    zipfile.ZipFile(partial)
    partial.replace(z)
    print("download finished", flush=True)
    zipfile.ZipFile(z).extractall(dest)

req = dest / "requirements.txt"
print(req.read_text())
!pip install -r "{req}"
# Colab's torchvision is built for an older torch. torch 2.14 needs 0.29 or
# importing qwen_omni_utils fails with "operator torchvision::nms does not exist".
!pip install torchvision==0.29.1
# 0.46.1 has no CUDA 13 binary. torch 2.14 looks for libbitsandbytes_cuda130.so.
!pip install bitsandbytes==0.50.2


files already present /content
yt-dlp==2026.10.21
torch==2.14.1
transformers==5.16.0
accelerate==1.15.0
qwen-vl-utils==0.0.14
qwen-omni-utils==0.0.9
bitsandbytes==0.46.1
librosa==1.0.0
tqdm==4.70.1
sentence-transformers==6.2.0
python-dotenv==1.2.1
pydantic==2.12.5
lm-format-enforcer==0.11.3

  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.50.2
    Uninstalling bitsandbytes-0.50.2:
      Successfully uninstalled bitsandbytes-0.50.2
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl (43.1 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.46.1
    Uninstalling bitsandbytes-0.46.1:
      Successfully uninstalled bitsandbytes-0.46.1


In [2]:
import array
import json
import math
import re
import subprocess
from pathlib import Path

import numpy as np
import torch
from qwen_omni_utils import process_mm_info
import transformers.tokenization_utils as _tokenizers
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
_tokenizers.PreTrainedTokenizerBase = PreTrainedTokenizerBase
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn
from pydantic import Field, TypeAdapter
from pydantic.dataclasses import dataclass
from tqdm.auto import tqdm
from transformers import AutoModelForTDT, AutoProcessor, BitsAndBytesConfig, Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

assert torch.cuda.is_available(), "Set the runtime to a T4 GPU, then rerun."


In [3]:
roots = [Path("/content/data/india_cricket"), Path("data/india_cricket")]
found = [path for root in roots if root.exists() for path in sorted(root.glob("*.mp4"))]
videos = []
for path in found:
    if any(path.resolve() == kept.resolve() for kept in videos):
        continue
    videos.append(path)
assert videos, "No mp4 files under data/india_cricket"
for video in videos:
    print(video)


/content/data/india_cricket/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p.mp4
/content/data/india_cricket/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p.mp4
/content/data/india_cricket/3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p.mp4


In [4]:
MODEL = "Qwen/Qwen2.5-Omni-3B"
BATCH = 8
LIMITS = (30, 45)
MIN_SECONDS = 12
SCENE_THRESHOLD = 0.3
PAUSE_DB = -32
PAUSE_SECONDS = 0.8
EDGE = 0.4
PEAK_RATIO = 1.25
PEAK_GAP = 20
PEAK_LEAD = 8
PEAK_TAIL = 12
COMMENT_AFTER = 28
ASR = "nvidia/parakeet-tdt-0.6b-v3"
CHUNK = 90

@dataclass
class ClipScore:
    start: float = Field(description="Start of this window in the full video, in seconds.")
    end: float = Field(description="End of this window in the full video, in seconds.")
    importance_score: float = Field(ge=0, le=1, description="Score only the commentary word. four, six, boundary, or edge is 0.5. wicket, caught, bowled, lbw, stumped, run out, fifty, or hundred is 0.65. 1 if this ball wins or loses the match, or the commentary says the match is won or lost. 0 is the toss, music, or a replay intro. 0.3 is a routine ball. Do not describe a different event.")
    desc: str = Field(max_length=120, description="One short sentence that repeats the commentary word and names the player.")
    importance_score_rationale: str = Field(max_length=160, description="Exactly: heard <commentary word>, so the score is <importance_score>.")

SCORE = TypeAdapter(ClipScore)
PROMPT = (
    "You are a cricket broadcast editor. The last sentence names the commentary word. Score only that word. "
    "Do not describe a different event. Ignore how excited the commentator sounds. "
    "A 30 second window and a 45 second window with the same commentary word get the same importance_score. "
    "Return one JSON object with start, end, importance_score, desc, and importance_score_rationale. "
    "Write importance_score first. Use only 0, 0.3, 0.5, 0.65, or 1. "
    "If the commentary word is four, six, boundary, or edge, importance_score is 0.5. "
    "If it is wicket, caught, bowled, lbw, stumped, run out, fifty, or hundred, importance_score is 0.65. "
    "If the commentary says this ball wins or loses the match, or the match is won or lost, importance_score is 1. "
    "0 is the toss, music, or a replay intro. 0.3 is a routine ball with none of those outcomes. "
    "desc repeats the commentary word and names the player. "
    "importance_score_rationale is: heard <word>, so the score is <number>."
)


### Edges

Shot cuts, then pauses, then loudness. One ffmpeg job at a time. Cuts and pauses are saved under `outputs/sports` and reused on the next run.


In [5]:
def duration(video):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(video)],
        check=True, capture_output=True, text=True,
    )
    return float(out.stdout)


def _clock(stamp):
    clock, _, frac = stamp.strip().partition(".")
    hours, minutes, secs = [int(part) for part in clock.split(":")]
    return hours * 3600 + minutes * 60 + secs + (int(frac) / 10 ** len(frac) if frac else 0)


def _ffmpeg(cmd, seconds, desc):
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    lines, seen = [], 0.0
    bar = tqdm(total=seconds, desc=desc, unit="s")
    for line in proc.stderr:
        lines.append(line)
        if not line.startswith("out_time="):
            continue
        stamp = line.split("=", 1)[1].strip()
        if stamp in ("", "N/A"):
            continue
        now = min(_clock(stamp), seconds)
        if now > seen:
            bar.update(now - seen)
            seen = now
    code = proc.wait()
    if seen < seconds:
        bar.update(seconds - seen)
    bar.close()
    if code != 0:
        raise RuntimeError("".join(lines)[-500:])
    return lines


def _cached(path, key, build):
    if path.exists():
        print(f"reuse {path.parent.name}  {path.stem[:40]}")
        return json.loads(path.read_text())[key]
    value = build()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({key: value}))
    return value


def shot_cuts(video, seconds):
    path = Path("outputs/sports/shots") / f"{video.stem}.json"

    def build():
        lines = _ffmpeg(
            ["ffmpeg", "-hide_banner", "-nostats", "-stats_period", "0.5", "-progress", "pipe:2", "-i", str(video),
             "-vf", f"scale=320:-2,select='gt(scene,{SCENE_THRESHOLD})',metadata=print",
             "-an", "-f", "null", "-"],
            seconds, f"shots {video.stem[:40]}",
        )
        return [float(line.rsplit("pts_time:", 1)[1].split()[0]) for line in lines if "pts_time:" in line]

    return _cached(path, "cuts", build)


def pauses(video, seconds):
    path = Path("outputs/sports/pauses") / f"{video.stem}.json"

    def build():
        lines = _ffmpeg(
            ["ffmpeg", "-hide_banner", "-nostats", "-stats_period", "0.5", "-progress", "pipe:2", "-i", str(video),
             "-af", f"silencedetect=noise={PAUSE_DB}dB:d={PAUSE_SECONDS}",
             "-f", "null", "-"],
            seconds, f"pauses {video.stem[:40]}",
        )
        gaps, start = [], None
        for line in lines:
            if "silence_start:" in line:
                start = float(line.rsplit("silence_start:", 1)[1].split()[0])
            elif "silence_end:" in line and start is not None:
                gaps.append((start, float(line.split("silence_end:", 1)[1].split("|")[0])))
                start = None
        return gaps

    return _cached(path, "gaps", build)


def boundaries(seconds, cuts, gaps):
    points = [0.0, seconds, *cuts]
    for start, end in gaps:
        points.extend((start, end))
    points = sorted(point for point in points if 0 <= point <= seconds + 0.01)
    merged = [points[0]]
    for point in points[1:]:
        if point - merged[-1] >= EDGE:
            merged.append(min(point, seconds))
    if seconds - merged[-1] >= EDGE:
        merged.append(seconds)
    elif merged[-1] != seconds:
        merged[-1] = seconds
    return merged


def loudness(video, seconds):
    rate = 8000
    proc = subprocess.Popen(
        ["ffmpeg", "-v", "error", "-i", str(video), "-ac", "1", "-ar", str(rate), "-f", "s16le", "-"],
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
    )
    hop = rate * 2
    levels = []
    bar = tqdm(total=seconds, desc=f"audio {video.stem[:40]}", unit="s")
    while True:
        buf = proc.stdout.read(hop)
        if len(buf) < hop:
            break
        samples = array.array("h")
        samples.frombytes(buf)
        levels.append(math.sqrt(sum(sample * sample for sample in samples) / len(samples)))
        bar.update(1)
    proc.wait()
    if bar.n < seconds:
        bar.update(seconds - bar.n)
    bar.close()
    smooth = []
    for i in range(len(levels)):
        chunk = levels[i:i + 2]
        smooth.append(sum(chunk) / len(chunk))
    return smooth


def scan(video):
    seconds = duration(video)
    cuts = shot_cuts(video, seconds)
    gaps = pauses(video, seconds)
    levels = loudness(video, seconds)
    return {"seconds": seconds, "bounds": boundaries(seconds, cuts, gaps), "levels": levels}



### Transcript

Parakeet transcribes after the scans. A surge is kept when an event word sits beside it.


In [6]:
def _median(values):
    ordered = sorted(values)
    return ordered[len(ordered) // 2]


def _peaks(levels):
    found = []
    for t in range(1, len(levels) - 1):
        base = _median(levels[max(0, t - 10):t + 11])
        if levels[t] < levels[t - 1] or levels[t] < levels[t + 1]:
            continue
        if base <= 0 or levels[t] < base * PEAK_RATIO:
            continue
        found.append((t, levels[t] - base))
    found.sort(key=lambda item: item[1], reverse=True)
    kept = []
    for t, contrast in found:
        if all(abs(t - other) >= PEAK_GAP for other, _ in kept):
            kept.append((t, contrast))
    return sorted(kept)


def _around(bounds, seconds, t, limit):
    end_cap = min(seconds, t + PEAK_TAIL)
    ends = [bound for bound in bounds if t <= bound <= end_cap + 0.05]
    if ends:
        end = ends[-1]
    else:
        end = min(bounds, key=lambda bound: abs(bound - (t + PEAK_TAIL)))
        if end < t:
            end = min(bound for bound in bounds if bound >= t)
    starts = [bound for bound in bounds if end - limit - 0.05 <= bound <= t + 0.05]
    if not starts:
        return None
    start = starts[0]
    if end - start < MIN_SECONDS:
        return None
    return round(start, 3), round(min(end, seconds), 3)


EVENTS = {
    "four", "fours", "six", "sixes", "boundary", "boundaries",
    "wicket", "wickets", "bowled", "caught", "catch", "lbw", "stumped",
    "fifty", "hundred", "century", "edged", "edge", "gone", "stumps", "run out",
}


def _plain(word):
    return "".join(ch for ch in word.lower() if ch.isalnum())


def _events(words):
    clean = [_plain(word["word"]) for word in words]
    found = []
    for i, token in enumerate(clean):
        nxt = clean[i + 1] if i + 1 < len(clean) else ""
        if token == "hundred" and nxt in {"matches", "match"}:
            continue
        cue = "run out" if token == "run" and nxt == "out" else token
        if cue not in EVENTS:
            continue
        moment = (words[i]["start"] + words[i]["end"]) / 2
        found.append((moment, cue))
    return found


def _words_from(tokens, offset):
    words, text, start, end = [], "", None, None
    for token in tokens:
        piece = token["token"]
        if text and piece[:1].isspace():
            words.append({"word": text, "start": round(start + offset, 3), "end": round(end + offset, 3)})
            text, start = "", None
        if start is None:
            start = token["start"]
        text += piece.lstrip(" ")
        end = token["end"]
    if text:
        words.append({"word": text, "start": round(start + offset, 3), "end": round(end + offset, 3)})
    return words


def transcribe(video, asr, asr_processor):
    raw = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", str(video), "-ac", "1", "-ar", "16000", "-f", "f32le", "-"],
        check=True, capture_output=True,
    ).stdout
    audio = np.frombuffer(raw, dtype=np.float32).copy()
    seconds = len(audio) / 16000
    words = []
    starts = range(0, int(seconds), CHUNK)
    for start in tqdm(starts, desc=f"transcript {video.stem[:32]}", unit="chunk"):
        piece = audio[int(start * 16000):int((start + CHUNK + 2) * 16000)]
        if len(piece) < 16000:
            continue
        inputs = asr_processor(piece, sampling_rate=16000, return_tensors="pt").to("cuda")
        inputs["input_features"] = inputs["input_features"].to(torch.float16)
        with torch.inference_mode():
            output = asr.generate(**inputs, return_dict_in_generate=True, max_new_tokens=1024)
        _, tokens = asr_processor.decode(output.sequences, durations=output.durations, skip_special_tokens=True)
        for word in _words_from(tokens[0], start):
            if start and word["start"] < start + 2:
                continue
            words.append(word)
    return words


def promising(bounds, seconds, levels, words):
    found, seen = [], set()

    def add(start, end, limit, peak, cue):
        key = (start, end, limit)
        if key in seen:
            return
        seen.add(key)
        found.append({"start": start, "end": end, "max_len": limit, "peak": peak, "cue": cue})

    cues = _events(words)
    for t, _contrast in _peaks(levels):
        near = [item for item in cues if t - PEAK_LEAD <= item[0] <= t + PEAK_TAIL]
        if not near:
            continue
        follow = [item for item in cues if t - PEAK_LEAD <= item[0] <= t + COMMENT_AFTER]
        anchor = follow[-1][0]
        for limit in LIMITS:
            placed = _around(bounds, seconds, anchor, limit)
            if placed:
                add(placed[0], placed[1], limit, float(t), near[0][1])
    return found



def transcript_path(video):
    return Path("outputs/sports/transcripts") / f"{video.stem}.json"


def ensure_transcripts(videos):
    missing = [video for video in videos if not transcript_path(video).exists()]
    if not missing:
        return
    asr_processor = AutoProcessor.from_pretrained(ASR)
    asr = AutoModelForTDT.from_pretrained(ASR, dtype=torch.float16).cuda().eval()
    for video in missing:
        words = transcribe(video, asr, asr_processor)
        path = transcript_path(video)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps({"words": words}))
        print(f"{len(words):4} words  {video.name}")
    del asr, asr_processor
    torch.cuda.empty_cache()


### Score

Qwen scores the kept clips. Each batch is cut, then scored.


In [7]:
import audioread.ffdec
import librosa
import numpy as np

def _load_ffmpeg(src, sr, offset, duration):
    native = src.samplerate
    channels = src.channels
    start = int((offset or 0) * native)
    stop = None if duration is None else start + int(duration * native)
    pieces = []
    seen = 0
    try:
        for block in src:
            frames = np.frombuffer(block, dtype="<i2")
            if channels > 1:
                frames = frames.reshape(-1, channels).mean(axis=1)
            lo = max(0, start - seen)
            hi = len(frames) if stop is None else max(0, min(len(frames), stop - seen))
            if hi > lo:
                pieces.append(frames[lo:hi].copy())
            seen += len(frames)
            if stop is not None and seen >= stop:
                break
    finally:
        src.close()
    audio = np.concatenate(pieces).astype(np.float32) / 32768.0 if pieces else np.zeros(0, dtype=np.float32)
    if sr and sr != native and audio.size:
        audio = librosa.resample(audio, orig_sr=native, target_sr=sr)
    return audio, sr or native

if not getattr(librosa.load, "_reads_ffmpeg", False):
    _librosa_load = librosa.load

    def _load(path, sr=22050, offset=0.0, duration=None, **kwargs):
        if isinstance(path, audioread.ffdec.FFmpegAudioFile):
            return _load_ffmpeg(path, sr, offset, duration)
        return _librosa_load(path, sr=sr, offset=offset, duration=duration, **kwargs)

    _load._reads_ffmpeg = True
    librosa.load = _load

def cut(video, start, end, path):
    subprocess.run(
        ["ffmpeg", "-y", "-ss", f"{start:.3f}", "-to", f"{end:.3f}", "-i", str(video),
         "-vf", "scale=320:-2", "-c:v", "libx264", "-preset", "veryfast",
         "-c:a", "aac", "-ar", "16000", "-ac", "1", str(path)],
        check=True, capture_output=True,
    )
    return path


def _field(text, name):
    match = re.search(r'"%s"\s*:\s*"((?:\\.|[^"\\])*)' % name, text)
    if not match:
        return ""
    return match.group(1).replace('\\"', '"').replace("\\n", " ")


def _number(text, name):
    match = re.search(r'"%s"\s*:\s*(-?\d+(?:\.\d+)?)' % name, text)
    return float(match.group(1)) if match else None


def read_score(reply):
    raw = reply.strip()
    left, right = raw.find("{"), raw.rfind("}")
    if 0 <= left < right:
        try:
            score = SCORE.validate_json(raw[left:right + 1])
        except Exception:
            score = None
        else:
            value = min(1.0, max(0.0, float(score.importance_score)))
            return {
                "start": float(score.start),
                "end": float(score.end),
                "desc": score.desc,
                "importance_score_rationale": score.importance_score_rationale,
                "importance_score": value,
                "importance": value,
                "raw": raw,
            }
    value = _number(raw, "importance_score")
    value = 0.0 if value is None else min(1.0, max(0.0, value))
    return {
        "start": _number(raw, "start"),
        "end": _number(raw, "end"),
        "desc": _field(raw, "desc"),
        "importance_score_rationale": _field(raw, "importance_score_rationale"),
        "importance_score": value,
        "importance": value,
        "raw": raw,
    }


def score_batch(clips, paths):
    conversations = [[{"role": "user", "content": [
        {"type": "video", "video": path.resolve().as_uri(), "max_frames": 8},
        {"type": "text", "text": PROMPT + " This window runs from %.3f to %.3f seconds. The commentary word is %s. Score only that word." % (clip["start"], clip["end"], clip.get("cue") or "the outcome you hear")},
    ]}] for clip, path in zip(clips, paths)]
    texts = [processor.apply_chat_template(item, tokenize=False, add_generation_prompt=True) for item in conversations]
    audios, images, videos = process_mm_info(conversations, use_audio_in_video=True)
    inputs = processor(
        text=texts, audio=audios, images=images, videos=videos,
        padding=True, return_tensors="pt", use_audio_in_video=True,
    )
    inputs = inputs.to("cuda")
    schema = build_transformers_prefix_allowed_tokens_fn(
        processor.tokenizer, JsonSchemaParser(SCORE.json_schema())
    )
    out = model.thinker.generate(
        **inputs,
        max_new_tokens=1024,
        use_audio_in_video=True,
        prefix_allowed_tokens_fn=schema,
    )
    if isinstance(out, tuple):
        out = out[0]
    if hasattr(out, "sequences"):
        out = out.sequences
    replies = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    del inputs, out
    torch.cuda.empty_cache()
    return [read_score(reply) for reply in replies]


def score_group(clips, paths):
    try:
        return score_batch(clips, paths)
    except RuntimeError:
        torch.cuda.empty_cache()
        if len(paths) == 1:
            raise
        mid = len(paths) // 2
        return score_group(clips[:mid], paths[:mid]) + score_group(clips[mid:], paths[mid:])


def overlap(a, b):
    inter = min(a["end"], b["end"]) - max(a["start"], b["start"])
    if inter <= 0:
        return 0.0
    union = (a["end"] - a["start"]) + (b["end"] - b["start"]) - inter
    return inter / union


def keep(clips):
    chosen = []
    for clip in sorted(clips, key=lambda item: item["importance"], reverse=True):
        if any(overlap(clip, other) > 0.2 for other in chosen):
            continue
        chosen.append(clip)
    return chosen


def save(clips, path, video):
    ready = [clip for clip in clips if "importance" in clip]
    payload = {
        "video": video.name,
        "clips": ready,
        "kept": {str(limit): keep([clip for clip in ready if clip["max_len"] == limit]) for limit in LIMITS},
    }
    path.write_text(json.dumps(payload, indent=2))


def cut_group(video, group, tag):
    return [cut(video, clip["start"], clip["end"], Path(f"/tmp/clip_{tag}_{i}.mp4")) for i, clip in enumerate(group)]


def push_sports(root=None):
    import urllib.parse
    import urllib.request
    root = Path(root or "outputs/sports")
    host = url.rsplit("/", 1)[0]
    saved = []
    for path in sorted(item for item in root.rglob("*") if item.is_file()):
        rel = path.relative_to(root).as_posix()
        target = f"{host}/sports/{urllib.parse.quote(rel)}"
        request = urllib.request.Request(target, data=path.read_bytes(), method="PUT")
        with urllib.request.urlopen(request, timeout=120) as response:
            print(f"saved outputs/sports/{rel}  {response.status}")
            saved.append(rel)
    if not saved:
        print("nothing to save under", root)
    return saved


def score_video(video, ready):
    words = json.loads(transcript_path(video).read_text())["words"]
    clips = promising(ready["bounds"], ready["seconds"], ready["levels"], words)
    counts = {limit: sum(clip["max_len"] == limit for clip in clips) for limit in LIMITS}
    by_span, seen = {}, []
    for clip in clips:
        key = (clip["start"], clip["end"])
        by_span.setdefault(key, []).append(clip)
        if len(by_span[key]) == 1:
            seen.append({"start": key[0], "end": key[1], "cue": clip.get("cue")})
    moments = len({clip["peak"] for clip in clips})
    print(f"{len(seen):3} scored   {moments} moments  {counts[30]} x 30s  {counts[45]} x 45s  {video.name}")
    out = Path("outputs/sports") / f"{video.stem}_clips.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    bar = tqdm(total=len(seen), unit="window")
    for index_start in range(0, len(seen), BATCH):
        group = seen[index_start:index_start + BATCH]
        paths = cut_group(video, group, index_start)
        tqdm.write(f"scoring {index_start + 1}-{index_start + len(group)} of {len(seen)}")
        for clip, item in zip(group, score_group(group, paths)):
            tqdm.write(f"{item['importance']:.2f}  {clip['start']:.0f}-{clip['end']:.0f}  {item['desc']}")
            item = dict(item)
            item.pop("start", None)
            item.pop("end", None)
            for twin in by_span[(clip["start"], clip["end"])]:
                twin.update(item)
            bar.update(1)
        save(clips, out, video)
    if not seen:
        save(clips, out, video)
    bar.close()
    push_sports()


In [8]:
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL, device_map="auto", attn_implementation="sdpa",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    enable_audio_output=False,
)
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1442 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                                                                      | Status     |  | 
---------------------------------------------------------------------------------------------------------+------------+--+-
token2wav.code2wav_bigvgan_model.resblocks.{0...17}.activations.{0, 1, 2, 3, 4, 5}.act.beta              | UNEXPECTED |  | 
talker.model.layers.{0...23}.self_attn.k_proj.weight                                                     | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_out.0.weight                            | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.attn.to_out.0.bias                              | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_blocks.{0...21}.ff.ff.{0, 3}.bias                               | UNEXPECTED |  | 
token2wav.code2wav_dit_model.transformer_b

In [9]:
push_sports()


saved outputs/sports/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p_clips.json  200
saved outputs/sports/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p_clips.json  200
saved outputs/sports/3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p_clips.json  200
saved outputs/sports/pauses/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p.json  200
saved outputs/sports/pauses/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p.json  200
saved outputs/sports/pauses/3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p.json  200
saved outputs/sports/shots/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p.json  20

["1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p_clips.json",
 '2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p_clips.json',
 '3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p_clips.json',
 "pauses/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p.json",
 'pauses/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p.json',
 'pauses/3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p.json',
 "shots/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p.json",
 'shots/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p.json',
 'shots/3.

In [10]:
ensure_transcripts(videos)
for video in videos:
    score_video(video, scan(video))


reuse shots  1.Sanju Samson's 67 Sparks Victory | 1st
reuse pauses  1.Sanju Samson's 67 Sparks Victory | 1st


audio 1.Sanju Samson's 67 Sparks Victory | 1st:   0%|          | 0/1112.746667 [00:00<?, ?s/s]

 34 scored   18 moments  18 x 30s  18 x 45s  1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p.mp4


  0%|          | 0/34 [00:00<?, ?window/s]

scoring 1-8 of 34


...

scoring 1-8 of 22
0.65  71-100  that is a brilliant start for Tim Sow
0.65  55-100  six
0.30  118-146  gone
0.30  103-146  gone
0.30  143-167  gone
0.30  136-167  it's gone
0.65  238-262  that's what's rock
0.50  223-262  that's picked up beautifully
scoring 9-16 of 22
0.65  262-292  that is very very clever
0.30  252-292  that's clever
0.65  359-385  clears the ground with ease
0.30  343-385  boundary
0.65  557-580  catch
0.65  535-580  catch
0.65  593-623  catch
0.65  580-623  catch
scoring 17-22 of 22
0.65  628-658  a flat fast six
0.65  615-658  a flat fast six
0.65  726-750  abses at leg side
0.65  708-750  the commentator says this ball wins or loses the match, or the match is won or lost
0.50  806-826  boundary
0.30  782-826  boundary
saved outputs/sports/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p_clips.json  200
saved outputs/sports/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | Ind

audio 3.India v USA | Match Highlights | Men’s:   0%|          | 0/419.86322 [00:00<?, ?s/s]

 20 scored   10 moments  10 x 30s  10 x 45s  3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p.mp4


  0%|          | 0/20 [00:00<?, ?window/s]

scoring 1-8 of 20
0.65  16-40  that's a big wicket
0.65  0-40  clear the boundary
0.30  30-60  boundary
0.65  16-60  that's a big wicket
0.65  84-101  wicket
0.65  60-101  wicket
0.30  169-197  boundary
0.65  158-197  that's the gap he looks for
scoring 9-16 of 20
0.30  189-217  six
0.65  175-217  six
0.65  269-285  that's a massive edge
0.30  242-285  hit that ball
0.30  299-326  he has scored a six
0.30  285-326  six
0.65  361-384  wickets
0.65  342-384  wickets
scoring 17-20 of 20
0.65  377-402  gone
0.65  361-402  gone
0.30  384-414  gone
0.30  377-414  gone
saved outputs/sports/1.Sanju Samson's 67 Sparks Victory | 1st T20I | English | Highlights | India Tour of New Zealand 2026_Sony LIV_720p_clips.json  200
saved outputs/sports/2.Tilak Varma Finishes It In Style | 2nd T20I | English | Highlights | India Tour of New Zealand 2026_Sony Sports Network_720p_clips.json  200
saved outputs/sports/3.India v USA | Match Highlights | Men’s T20 World Cup 2026_ICC_720p_clips.json  200
saved ou